<a href="https://colab.research.google.com/github/Caleb-John1603/AI_5th_sem/blob/main/Cuckoo%20Search%201BM24CS095.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import math

np.random.seed(42)

tasks = [
    {"id": i, "length": np.random.randint(100, 1000), "cpu_req": np.random.randint(1, 4), "mem_req": np.random.randint(1, 8)}
    for i in range(20)
]

vms = [
    {"id": 0, "mips": 1000, "cpu_cap": 8,  "mem_cap": 16, "cost_rate": 0.15},
    {"id": 1, "mips": 1500, "cpu_cap": 12, "mem_cap": 32, "cost_rate": 0.25},
    {"id": 2, "mips": 2000, "cpu_cap": 16, "mem_cap": 64, "cost_rate": 0.40},
    {"id": 3, "mips": 800,  "cpu_cap": 4,  "mem_cap": 8,  "cost_rate": 0.08},
]

num_tasks = len(tasks)
num_vms = len(vms)

def calculate_fitness(nest):
    vm_execution_times = np.zeros(num_vms)
    vm_cpu_usage = np.zeros(num_vms)
    vm_mem_usage = np.zeros(num_vms)
    total_cost = 0.0

    for task_idx, vm_idx in enumerate(nest):
        vm_idx = int(np.clip(round(vm_idx), 0, num_vms - 1))
        task = tasks[task_idx]
        vm = vms[vm_idx]

        exec_time = task["length"] / vm["mips"]
        vm_execution_times[vm_idx] += exec_time

        total_cost += exec_time * vm["cost_rate"]

        vm_cpu_usage[vm_idx] += task["cpu_req"]
        vm_mem_usage[vm_idx] += task["mem_req"]

    makespan = np.max(vm_execution_times)

    penalty = 0
    for i in range(num_vms):
        if vm_cpu_usage[i] > vms[i]["cpu_cap"]:
            penalty += (vm_cpu_usage[i] - vms[i]["cpu_cap"]) * 10000
        if vm_mem_usage[i] > vms[i]["mem_cap"]:
            penalty += (vm_mem_usage[i] - vms[i]["mem_cap"]) * 10000

    weight_time = 0.7
    weight_cost = 0.3
    fitness_score = (weight_time * makespan) + (weight_cost * total_cost) + penalty
    return fitness_score

def levy_flight(step_size=0.01, beta=1.5):
    num = math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
    den = math.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2))
    sigma = (num / den) ** (1 / beta)

    u = np.random.normal(0, sigma, size=num_tasks)
    v = np.random.normal(0, 1, size=num_tasks)

    step = u / (np.abs(v) ** (1 / beta))
    return step_size * step

def cuckoo_search_cloud(num_nests=25, max_iter=100, pa=0.25):
    nests = np.random.uniform(0, num_vms - 1, size=(num_nests, num_tasks))
    fitness = np.array([calculate_fitness(n) for n in nests])

    best_idx = np.argmin(fitness)
    best_nest = nests[best_idx].copy()
    best_fitness = fitness[best_idx]

    history = []

    for iteration in range(max_iter):
        random_nest_idx = np.random.randint(0, num_nests)
        current_nest = nests[random_nest_idx]

        step = levy_flight()
        new_nest = current_nest + step * (current_nest - best_nest)
        new_nest = np.clip(new_nest, 0, num_vms - 1)

        new_fitness = calculate_fitness(new_nest)

        target_idx = np.random.randint(0, num_nests)
        if new_fitness < fitness[target_idx]:
            nests[target_idx] = new_nest
            fitness[target_idx] = new_fitness

        for i in range(num_nests):
            if np.random.rand() < pa:
                r1, r2 = np.random.randint(0, num_nests, size=2)
                nests[i] = nests[i] + np.random.rand() * (nests[r1] - nests[r2])
                nests[i] = np.clip(nests[i], 0, num_vms - 1)
                fitness[i] = calculate_fitness(nests[i])

        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

        history.append(best_fitness)

    return np.round(best_nest).astype(int), best_fitness, history

best_allocation, final_fitness, convergence_history = cuckoo_search_cloud()

print(f"Optimization Successful! Best Unified Fitness Score: {final_fitness:.4f}\n")
print("Optimal Task-to-VM Assignment Strategy:")
for task_id, vm_id in enumerate(best_allocation):
    print(f" 🔹 Task {task_id:02d} (Length: {tasks[task_id]['length']:3d}) ──> Assigned to VM {vm_id}")


Optimization Successful! Best Unified Fitness Score: 10002.6157

Optimal Task-to-VM Assignment Strategy:
 🔹 Task 00 (Length: 202) ──> Assigned to VM 2
 🔹 Task 01 (Length: 206) ──> Assigned to VM 1
 🔹 Task 02 (Length: 714) ──> Assigned to VM 1
 🔹 Task 03 (Length: 314) ──> Assigned to VM 1
 🔹 Task 04 (Length: 187) ──> Assigned to VM 2
 🔹 Task 05 (Length: 971) ──> Assigned to VM 2
 🔹 Task 06 (Length: 408) ──> Assigned to VM 1
 🔹 Task 07 (Length: 513) ──> Assigned to VM 0
 🔹 Task 08 (Length: 291) ──> Assigned to VM 3
 🔹 Task 09 (Length: 559) ──> Assigned to VM 1
 🔹 Task 10 (Length: 352) ──> Assigned to VM 2
 🔹 Task 11 (Length: 574) ──> Assigned to VM 2
 🔹 Task 12 (Length: 781) ──> Assigned to VM 2
 🔹 Task 13 (Length: 786) ──> Assigned to VM 1
 🔹 Task 14 (Length: 975) ──> Assigned to VM 0
 🔹 Task 15 (Length: 931) ──> Assigned to VM 0
 🔹 Task 16 (Length: 584) ──> Assigned to VM 0
 🔹 Task 17 (Length: 120) ──> Assigned to VM 2
 🔹 Task 18 (Length: 373) ──> Assigned to VM 2
 🔹 Task 19 (Length: 1